# Phase 0: Historical data validation

## Can SofaScore support the four-role analysis for the 1986 World Cup?

Before designing the full pipeline, this project needs to know whether a source
like SofaScore has match-level statistical detail for older World Cups that is
comparable to what's available for modern ones. If 1986 data is too thin,
that changes how (or whether) Maradona can participate in the comparison.

Test case: Argentina vs England, 1986 World Cup quarter-final (Maradona's
"Hand of God" and "Goal of the Century" match), chosen because it's his most
documented match, i.e. the best case for data availability, not the average
case.

---

## Fase 0: validación de datos históricos

## ¿SofaScore puede sostener el análisis de los cuatro roles para el Mundial de 1986?

Antes de diseñar el pipeline completo, el proyecto necesita saber si una fuente
como SofaScore tiene el mismo nivel de detalle estadístico a nivel de partido
para Mundiales antiguos que para los modernos. Si los datos de 1986 son
demasiado pobres, eso cambia cómo (o si) Maradona puede participar en la
comparación.

Caso de prueba: Argentina vs Inglaterra, cuartos de final del Mundial 1986
(el partido de "la Mano de Dios" y el "Gol del Siglo" de Maradona), elegido
por ser su partido más documentado, es decir, el mejor escenario posible
para disponibilidad de datos, no el escenario promedio.


In [1]:
import pandas as pd

lineups_1986 = pd.read_csv("../data/raw/sofascore_1986_argentina_england_lineups.csv")
maradona = lineups_1986[lineups_1986["name"].str.contains("Maradona", na=False)]

role_columns = {
    "finisher": ["goals", "totalShots", "onTargetScoringAttempt"],
    "dribbler": ["totalContest", "wonContest", "wasFouled"],
    "creator": ["keyPass", "goalAssist", "accurateCross"],
    "organizer": ["totalPass", "accuratePass", "touches"],
}

for role, cols in role_columns.items():
    print(f"{role}:")
    print(maradona[cols].to_string(index=False))
    print()


finisher:
 goals  totalShots  onTargetScoringAttempt
   2.0           7                     3.0

dribbler:
 totalContest  wonContest  wasFouled
         15.0        10.0        7.0

creator:
 keyPass  goalAssist  accurateCross
     5.0         0.0            1.0

organizer:
 totalPass  accuratePass  touches
      31.0          24.0     73.0



**Result:** the 1986 match, extracted live from SofaScore's internal API
(event id `7846755`), returns real per-player values for all four candidate
role dimensions. Maradona's line: 2 goals, 7 shots, 10 of 15 successful
dribbles, 5 key passes, 31 passes with 24 completed, 73 touches, a 9.8
match rating. This is the same level of detail we'd expect from a modern
match, not a reconstructed summary with just goals and cards.

---

**Resultado:** el partido de 1986, extraído en vivo de la API interna de
SofaScore (id de evento `7846755`), devuelve valores reales por jugador para
las cuatro dimensiones candidatas de rol. La línea de Maradona: 2 goles,
7 remates, 10 de 15 regates exitosos, 5 pases clave, 31 pases con 24
completados, 73 toques, nota de partido 9.8. Es el mismo nivel de detalle
que esperaríamos de un partido moderno, no un resumen reconstruido con
solo goles y tarjetas.


## An open question the data raised: NaN vs. zero

Several event-derived columns (`fouls`, `hitWoodwork`, `bigChanceMissed`)
return `NaN` instead of `0` for some players who clearly played significant
minutes, including Maradona himself (`fouls: NaN` despite being heavily
involved in duels). Before assuming this is a 1986-specific coverage gap,
it needs to be checked against a modern match. If the same pattern shows up
in a very well-documented recent match, it's just how the API represents
"no event of this type recorded," not a sign of weaker historical data.

---

## Una pregunta abierta que surgió de los datos: NaN vs. cero

Varias columnas derivadas de eventos (`fouls`, `hitWoodwork`,
`bigChanceMissed`) devuelven `NaN` en vez de `0` para algunos jugadores que
claramente jugaron minutos significativos, incluyendo al propio Maradona
(`fouls: NaN` a pesar de estar muy involucrado en duelos). Antes de asumir
que esto es una brecha de cobertura específica de 1986, hay que revisarlo
contra un partido moderno. Si el mismo patrón aparece en un partido reciente
muy bien documentado, es simplemente cómo la API representa "no se registró
ningún evento de este tipo", no una señal de datos históricos más débiles.


In [2]:
lineups_2022 = pd.read_csv("../data/raw/sofascore_2022_argentina_france_final_lineups.csv")

played_45_plus = lineups_2022[lineups_2022["minutesPlayed"].fillna(0) >= 45]
missing_fouls = played_45_plus[played_45_plus["fouls"].isna()]

print(f"{len(missing_fouls)} of {len(played_45_plus)} players with 45+ minutes "
      f"in the 2022 final also have fouls = NaN")
missing_fouls[["name", "fouls", "wasFouled", "duelWon", "duelLost", "minutesPlayed"]]


8 of 25 players with 45+ minutes in the 2022 final also have fouls = NaN


,name,fouls,wasFouled,duelWon,duelLost,minutesPlayed
0,Emiliano Martínez,NaN,NaN,NaN,NaN,120.0
1,Nahuel Molina,NaN,NaN,1.0,3.0,90.0
7,Alexis Mac Allister,NaN,3.0,8.0,8.0,116.0
10,Ángel Di María,NaN,3.0,7.0,5.0,64.0
26,Hugo Lloris,NaN,1.0,1.0,NaN,120.0
28,Raphaël Varane,NaN,1.0,1.0,NaN,113.0
34,Antoine Griezmann,NaN,1.0,6.0,3.0,71.0
39,Kingsley Coman,NaN,1.0,6.0,5.0,49.0


**Result:** 8 of 25 players with 45+ minutes in the 2022 World Cup final,
including well known and heavily-involved players like Di Maria and
Griezmann, also show `fouls: NaN`. This match has some of the best coverage
in SofaScore's dataset, so the pattern isn't caused by weak historical
reconstruction. It's a general convention of this API: these fields most
likely come from an event feed, and a stat key only gets created when at
least one event of that type is recorded for a player. No key means no
event, not "not tracked because it's an old match."

**Decision:** event-count fields (`fouls`, `bigChanceMissed`,
`hitWoodwork`, and similarly structured columns) will be treated as `0`
when `NaN` during data cleaning, applied consistently across all eras.
This is a data-cleaning assumption, not a proven fact about each individual
player-match, and it's logged here and in `METHODOLOGY.md` so it can be
revisited if later evidence contradicts it.

## Phase 0 conclusion

SofaScore's internal API provides match-level data for the 1986 World Cup
at a level of detail comparable to 2022, covering candidate metrics for all
four functional roles. Maradona is cleared to participate in the full
four-role analysis, not just basic stats. This was tested against his single
most-documented match as a best case, so per-match coverage should still be
spot-checked for less prominent 1986 matches once the full extraction is
built.

---

**Resultado:** 8 de 25 jugadores con 45+ minutos en la final del Mundial
2022, incluyendo jugadores muy conocidos y muy involucrados como Di María y
Griezmann, también tienen `fouls: NaN`. Este partido tiene una de las
mejores coberturas del dataset de SofaScore, así que el patrón no lo causa
una reconstrucción histórica débil. Es una convención general de esta API:
lo más probable es que estos campos vengan de un feed de eventos, y una
clave de estadística solo se crea cuando hay al menos un evento de ese tipo
registrado para un jugador. Sin clave no hay evento, no significa "no se
registró por ser un partido viejo".

**Decisión:** los campos de conteo de eventos (`fouls`, `bigChanceMissed`,
`hitWoodwork`, y columnas con estructura similar) se van a tratar como `0`
cuando sean `NaN` durante la limpieza de datos, aplicado igual para todas
las épocas. Esto es un supuesto de limpieza de datos, no un hecho probado
sobre cada partido-jugador individual, y queda registrado acá y en
`METHODOLOGY.md` para poder revisarlo si aparece evidencia que lo contradiga.

## Conclusión de la Fase 0

La API interna de SofaScore ofrece datos a nivel de partido para el Mundial
de 1986 con un nivel de detalle comparable al de 2022, cubriendo métricas
candidatas para los cuatro roles funcionales. Maradona queda habilitado para
participar en el análisis completo de los cuatro roles, no solo en
estadísticas básicas. Esto se probó contra su partido más documentado como
mejor escenario posible, así que la cobertura por partido todavía debería
revisarse puntualmente para partidos menos prominentes de 1986 una vez que
se construya la extracción completa.
